In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

## Quick Start

1. Copy the `rent_listings.csv.template` and populate it with relevant data
2. Run this notebook

In [ ]:
data_dir = "data/rentals/"
files = os.listdir(data_dir)
files = [f for f in files if f.endswith('.csv')]
files

In [ ]:
ind = -1
file = files[ind]
df_raw = pd.read_csv(os.path.join(data_dir, file))
df_raw.head()

In [ ]:
df_raw.info()

## Wrangle data

In [ ]:
df = df_raw.copy()
df["rooms"] = df["rooms"].replace(pd.NA, -1)
df["floor"] = df["floor"].apply(lambda x: x.split("/")[0] if isinstance(x, str) else -1).astype(int)
df["year"] = df["year"].replace(np.nan, -1).astype(int)
df["rent_per_m2"] = df["rent_per_month_eur"] / df["square_meters"]
df["post_2000"] = df["year"].apply(lambda x: 1 if x >= 2000 else 0)
df.head(2)

In [ ]:
df.info()

### Visualise correlations

In [ ]:
cols_numeric = df.select_dtypes(include=[np.number]).columns.tolist()
cols_numeric

In [ ]:
df_corr = df[
   cols_numeric 
].corr()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_corr, annot=True, cmap='coolwarm', center=0, square=True, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(15, 15))
pd.plotting.scatter_matrix(df[cols_numeric], figsize=(15, 15))

plt.suptitle('Scatter Matrix of Numeric Features')
plt.tight_layout()
plt.show()

## Visualise over time

In [ ]:
quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]
rent_quantiles = df['rent_per_month_eur'].quantile(quantiles)
rent_sqm_quantiles = df['rent_per_m2'].quantile(quantiles)
for q, rent_q, rent_sqm_q in zip(quantiles, rent_quantiles, rent_sqm_quantiles):
    print(f"{int(q*100)}th percentile rent per month: {rent_q:.2f} EUR, rent per square meter: {rent_sqm_q:.2f} EUR")

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df,
    x=list(range(0, df.shape[0])),
    y='rent_per_month_eur',
    hue="post_2000"
)

plt.axhline(y=rent_quantiles[0.5], color='r', linestyle='--', label=f'Median Rent per Month {rent_quantiles[0.5]:.2f} EUR')
plt.axhline(y=rent_quantiles[0.1], color='g', linestyle='--', label=f'10th Percentile Rent per Month {rent_quantiles[0.1]:.2f} EUR')

plt.title('Rent per month, chronological order (oldest first)')

plt.legend(loc='lower right')
plt.xlabel('Listing Index (chronological)')
plt.ylabel('Rent per Month (€)')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.barplot(
    data=df,
    x=list(range(0, df.shape[0])),
    y='rent_per_m2',
    hue="post_2000"
)
plt.axhline(y=rent_sqm_quantiles[0.5], color='r', linestyle='--', label=f'Median Rent per Square Meter {rent_sqm_quantiles[0.5]:.2f} EUR')
plt.axhline(y=rent_sqm_quantiles[0.1], color='g', linestyle='--', label=f'10th Percentile Rent per Square Meter {rent_sqm_quantiles[0.1]:.2f} EUR')

plt.title('Rent per square meter, chronological order (oldest first)')
plt.legend(loc='lower right')
plt.xlabel('Listing Index (chronological)')
plt.ylabel('Rent per Square Meter (€/m2)')
plt.tight_layout()
plt.show()

In [ ]:
mask_cheapest_sqm = df['rent_per_m2'] <= rent_sqm_quantiles[0.1]
df_cheapest_sqm = df[mask_cheapest_sqm]
df_cheapest_sqm

In [ ]:
df_cheapest_sqm.describe()[['rent_per_month_eur', 'rent_per_m2', 'square_meters', 'rooms', 'floor']].iloc[[1,2,3,-1]]